# RAG Ingestion Pipeline
## Petunjuk Teknis Budidaya Tanaman Sayuran

Pipeline ini membaca PDF, membuat chunk teks, menghasilkan embedding multilingual,
dan menyimpannya ke ChromaDB.

**Urutan eksekusi:**
1. Install dependencies
2. Upload PDF ke Colab
3. Ekstrak dan chunk teks
4. Build ChromaDB vector store
5. Validasi hasil ingestion
6. Export folder `chroma_db/` ke Google Drive

In [1]:
# ── CELL 1: Install dependencies (versi pinned untuk Colab) ──────────────────
!pip install -q -U pip
!pip install -q \
    qdrant-client==1.9.1 \
    sentence-transformers==2.7.0 \
    transformers==4.39.3 \
    tokenizers==0.15.2 \
    huggingface-hub==0.23.4 \
    pypdf==5.1.0 \
    pdfplumber==0.11.4 \
    langdetect==1.0.9

print("\n✅ Install selesai.")
print("⚠️  PENTING: Klik Runtime > Restart session, lalu lanjut ke Cell 2.")
print("   Jangan jalankan Cell 1 lagi setelah restart.")


✅ Install selesai.
⚠️  PENTING: Klik Runtime > Restart session, lalu lanjut ke Cell 2.
   Jangan jalankan Cell 1 lagi setelah restart.


In [2]:
# ── CELL 2: Upload PDF ────────────────────────────────────────────────────────
from google.colab import files

print("Upload file PDF petunjuk teknis budidaya sayuran:")
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
print(f"File tersimpan: {PDF_PATH}")

Upload file PDF petunjuk teknis budidaya sayuran:


Saving Petunjuk-Teknis-Budidaya-Tanaman-Sayuran.pdf to Petunjuk-Teknis-Budidaya-Tanaman-Sayuran.pdf
File tersimpan: Petunjuk-Teknis-Budidaya-Tanaman-Sayuran.pdf


In [3]:
# ── CELL 3: Ekstrak teks dari PDF ────────────────────────────────────────────
import pdfplumber
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path: str) -> list[dict]:
    """
    Ekstrak teks per halaman dari PDF.
    Fallback ke pypdf jika pdfplumber gagal pada halaman tertentu.
    Mengembalikan list of dict: {page_num, text, char_count}
    """
    pages = []
    reader_backup = PdfReader(pdf_path)

    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if not text or len(text.strip()) < 20:
                text = reader_backup.pages[i].extract_text() or ""
            text = text.strip()
            if text:
                pages.append({
                    "page_num": i + 1,
                    "text": text,
                    "char_count": len(text)
                })
    return pages

raw_pages = extract_text_from_pdf(PDF_PATH)
total_chars = sum(p["char_count"] for p in raw_pages)
print(f"Total halaman berhasil diekstrak : {len(raw_pages)}")
print(f"Total karakter                   : {total_chars:,}")
print(f"\nContoh halaman 1 (200 karakter pertama):")
print(raw_pages[0]["text"][:200] if raw_pages else "[Tidak ada teks]")

Total halaman berhasil diekstrak : 142
Total karakter                   : 187,552

Contoh halaman 1 (200 karakter pertama):
Petunjuk Teknis
Budidaya Tanaman Sayuran
Oleh :
Wiwin Setiawati, Rini Murtiningsih,
Gina Aliya Sopha dan Tri Handayani
BALAI PENELITIAN TANAMAN SAYURAN
PUSAT PENELITIAN DAN PENGEMBANGAN HORTIKULTURA
B


In [4]:
# ── CELL 4: Chunking dengan overlap ──────────────────────────────────────────
from langdetect import detect

CHUNK_SIZE = 500       # karakter per chunk
CHUNK_OVERLAP = 80     # overlap antar chunk untuk konteks

def chunk_text(text: str, page_num: int, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[dict]:
    """
    Membagi teks menjadi chunk dengan sliding window overlap.
    Mencoba memotong di batas kalimat (.!?) untuk kualitas lebih baik.
    """
    chunks = []
    start = 0
    chunk_idx = 0

    while start < len(text):
        end = start + chunk_size
        segment = text[start:end]

        if end < len(text):
            # Cari batas kalimat terdekat
            for delim in [". ", "! ", "? ", "\n"]:
                boundary = segment.rfind(delim)
                if boundary > chunk_size // 2:
                    segment = segment[:boundary + 1]
                    break

        segment = segment.strip()
        if len(segment) > 50:
            try:
                lang = detect(segment)
            except Exception:
                lang = "id"

            chunks.append({
                "chunk_id": f"page{page_num}_chunk{chunk_idx}",
                "text": segment,
                "page_num": page_num,
                "chunk_idx": chunk_idx,
                "char_count": len(segment),
                "lang": lang
            })
            chunk_idx += 1

        start += len(segment) - overlap if len(segment) > overlap else len(segment)

    return chunks

all_chunks = []
for page in raw_pages:
    chunks = chunk_text(page["text"], page["page_num"])
    all_chunks.extend(chunks)

print(f"Total chunk dibuat  : {len(all_chunks)}")
print(f"Rata-rata per halaman: {len(all_chunks)/len(raw_pages):.1f} chunk")
print(f"\nContoh chunk pertama:")
print(f"  ID   : {all_chunks[0]['chunk_id']}")
print(f"  Lang : {all_chunks[0]['lang']}")
print(f"  Text : {all_chunks[0]['text'][:200]}")

Total chunk dibuat  : 722
Rata-rata per halaman: 5.1 chunk

Contoh chunk pertama:
  ID   : page2_chunk0
  Lang : id
  Text : Petunjuk Teknis
Budidaya Tanaman Sayuran
Oleh :
Wiwin Setiawati, Rini Murtiningsih,
Gina Aliya Sopha dan Tri Handayani
BALAI PENELITIAN TANAMAN SAYURAN
PUSAT PENELITIAN DAN PENGEMBANGAN HORTIKULTURA
B


In [6]:
# ── CELL 5: Push ke Qdrant Cloud ─────────────────────────────────────────────
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer
import os

# ── Isi credentials Qdrant Cloud kamu di sini ─────────────────────────────────
from google.colab import userdata
QDRANT_URL     = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
# ──────────────────────────────────────────────────────────────────────────────

COLLECTION_NAME = "sayuran_kb"
EMBED_MODEL     = "paraphrase-multilingual-MiniLM-L12-v2"
EMBED_DIM       = 384

print(f"Memuat embedding model: {EMBED_MODEL}")
model = SentenceTransformer(EMBED_MODEL)
print("Model berhasil dimuat!")

# Koneksi ke Qdrant Cloud
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)
print("Terhubung ke Qdrant Cloud!")

# Hapus collection lama kalau ada
existing = [c.name for c in client.get_collections().collections]
if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)
    print(f"Collection lama '{COLLECTION_NAME}' dihapus.")

# Buat collection baru
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE)
)
print(f"Collection '{COLLECTION_NAME}' dibuat.")

# Embed semua chunk
print(f"\nMembuat embedding untuk {len(all_chunks)} chunk...")
texts = [c['text'] for c in all_chunks]
EMBED_BATCH = 64
all_embeddings = []
for i in range(0, len(texts), EMBED_BATCH):
    batch_emb = model.encode(texts[i:i+EMBED_BATCH], normalize_embeddings=True)
    all_embeddings.extend(batch_emb.tolist())
    print(f"  Embed batch {i//EMBED_BATCH+1}: {min(i+EMBED_BATCH, len(texts))}/{len(texts)}")
print("Embedding selesai!")

# Upload ke Qdrant Cloud
print(f"\nMengupload ke Qdrant Cloud...")
UPLOAD_BATCH = 100
total_uploaded = 0
for i in range(0, len(all_chunks), UPLOAD_BATCH):
    batch_chunks = all_chunks[i:i+UPLOAD_BATCH]
    batch_embs   = all_embeddings[i:i+UPLOAD_BATCH]
    points = [
        PointStruct(
            id=i+j,
            vector=emb,
            payload={
                "text":      c['text'],
                "page_num":  c['page_num'],
                "chunk_idx": c['chunk_idx'],
                "char_count":c['char_count'],
                "lang":      c['lang'],
            }
        )
        for j, (c, emb) in enumerate(zip(batch_chunks, batch_embs))
    ]
    client.upsert(collection_name=COLLECTION_NAME, points=points)
    total_uploaded += len(points)
    print(f"  Batch {i//UPLOAD_BATCH+1}: {total_uploaded}/{len(all_chunks)} chunk terupload")

# Verifikasi
info = client.get_collection(COLLECTION_NAME)
print(f"\n✅ Selesai! {info.points_count} chunk tersimpan di Qdrant Cloud.")
print(f"   Collection: {COLLECTION_NAME}")
print(f"   URL       : {QDRANT_URL}")

Memuat embedding model: paraphrase-multilingual-MiniLM-L12-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model berhasil dimuat!
Terhubung ke Qdrant Cloud!
Collection 'sayuran_kb' dibuat.

Membuat embedding untuk 722 chunk...
  Embed batch 1: 64/722
  Embed batch 2: 128/722
  Embed batch 3: 192/722
  Embed batch 4: 256/722
  Embed batch 5: 320/722
  Embed batch 6: 384/722
  Embed batch 7: 448/722
  Embed batch 8: 512/722
  Embed batch 9: 576/722
  Embed batch 10: 640/722
  Embed batch 11: 704/722
  Embed batch 12: 722/722
Embedding selesai!

Mengupload ke Qdrant Cloud...
  Batch 1: 100/722 chunk terupload
  Batch 2: 200/722 chunk terupload
  Batch 3: 300/722 chunk terupload
  Batch 4: 400/722 chunk terupload
  Batch 5: 500/722 chunk terupload
  Batch 6: 600/722 chunk terupload
  Batch 7: 700/722 chunk terupload
  Batch 8: 722/722 chunk terupload

✅ Selesai! 722 chunk tersimpan di Qdrant Cloud.
   Collection: sayuran_kb
   URL       : https://683797b0-230f-4e51-970f-18a15b3e6368.australia-southeast1-0.gcp.cloud.qdrant.io


In [7]:
# ── CELL 6: Validasi — query test retrieval dari Qdrant Cloud ────────────────

def test_retrieval(query: str, n_results: int = 2):
    query_vector = model.encode(query, normalize_embeddings=True).tolist()
    results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        limit=n_results,
        with_payload=True,
    )
    print(f"Query: '{query}'")
    print("─" * 60)
    for j, hit in enumerate(results):
        score = round(hit.score * 100, 1)
        text  = hit.payload.get('text', '')[:200]
        page  = hit.payload.get('page_num', '?')
        print(f"  [{j+1}] Halaman {page} | Relevansi: {score}%")
        print(f"       {text}...")
        print()

test_queries = [
    "cara menanam cabai",
    "penyakit tanaman tomat",
    "kebutuhan air bayam",
    "pupuk organik sayuran",
    "hama ulat pada kangkung"
]

for q in test_queries:
    test_retrieval(q)
    print("=" * 60)

Query: 'cara menanam cabai'
────────────────────────────────────────────────────────────
  [1] Halaman 75 | Relevansi: 59.5%
       tanaman dan pengasapan
menggunakan belerang.
Balai Penelitian Tanaman Sayuran 68...

  [2] Halaman 38 | Relevansi: 57.8%
       Petunjuk Teknis Prima Tani W. Setiawati, R. Murtiningsih,
G.A. Sopha, dan T. Handayati :
Budidaya Tanaman Sayuran
berhubungan dengan tempat tumbuh tanaman cabai (sawah atau
tegalan). Tanaman cabai yan...

Query: 'penyakit tanaman tomat'
────────────────────────────────────────────────────────────
  [1] Halaman 132 | Relevansi: 70.3%
       g tanaman agar tidak
roboh. Ajir dapat dibuat dari bambu dengan panjang 1–1,5 m. Tanaman
tomat diikatkan pada ajir tersebut secara longgar, sehingga tanaman
tersebut cukup leluasa berkembang.
8. Penge...

  [2] Halaman 116 | Relevansi: 70.3%
       a semiclausum)
- Tumpangsari petsai– tomat.
Balai Penelitian Tanaman Sayuran 109...

Query: 'kebutuhan air bayam'
───────────────────────────────────

In [8]:
# ── CELL 7: (Opsional) Cek info collection di Qdrant Cloud ──────────────────

info = client.get_collection(COLLECTION_NAME)
print("=" * 50)
print("  INFO COLLECTION QDRANT CLOUD")
print("=" * 50)
print(f"  Nama       : {COLLECTION_NAME}")
print(f"  Jumlah poin: {info.points_count}")
print(f"  Dimensi    : {info.config.params.vectors.size}")
print(f"  Distance   : {info.config.params.vectors.distance}")
print(f"  Status     : {info.status}")
print("=" * 50)
print("  Ingestion ke Qdrant Cloud SELESAI!")
print("  Sekarang set QDRANT_URL dan QDRANT_API_KEY")
print("  di Streamlit Cloud > Settings > Secrets")
print("=" * 50)

  INFO COLLECTION QDRANT CLOUD
  Nama       : sayuran_kb
  Jumlah poin: 722
  Dimensi    : 384
  Distance   : Cosine
  Status     : green
  Ingestion ke Qdrant Cloud SELESAI!
  Sekarang set QDRANT_URL dan QDRANT_API_KEY
  di Streamlit Cloud > Settings > Secrets


In [10]:
# ── CELL 8: Ringkasan ingestion ───────────────────────────────────────────────

lang_counts = {}
for c in all_chunks:
    lang_counts[c["lang"]] = lang_counts.get(c["lang"], 0) + 1

print("=" * 50)
print("  RINGKASAN INGESTION")
print("=" * 50)
print(f"  PDF          : {PDF_PATH}")
print(f"  Halaman      : {len(raw_pages)}")
print(f"  Chunk        : {len(all_chunks)}")
print(f"  Chunk size   : {CHUNK_SIZE} karakter (overlap {CHUNK_OVERLAP})")
print(f"  Embedding    : {EMBED_MODEL}")
print(f"  Vector store : Qdrant Cloud")
print(f"  Collection   : {COLLECTION_NAME}")
print(f"  Distribusi bahasa:")
for lang, count in sorted(lang_counts.items(), key=lambda x: -x[1]):
    pct = count / len(all_chunks) * 100
    print(f"    {lang:6s}: {count:4d} chunk ({pct:.1f}%)")
print("=" * 50)
print("  Ingestion SELESAI. Siap untuk app.py")
print("=" * 50)

  RINGKASAN INGESTION
  PDF          : Petunjuk-Teknis-Budidaya-Tanaman-Sayuran.pdf
  Halaman      : 142
  Chunk        : 722
  Chunk size   : 500 karakter (overlap 80)
  Embedding    : paraphrase-multilingual-MiniLM-L12-v2
  Vector store : Qdrant Cloud
  Collection   : sayuran_kb
  Distribusi bahasa:
    id    :  720 chunk (99.7%)
    tl    :    1 chunk (0.1%)
    en    :    1 chunk (0.1%)
  Ingestion SELESAI. Siap untuk app.py
